<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c34-header.png" alt="Nextia Learning · Anomaly Detection: Find What Is Unusual" width="100%">

# Forecast and compare

**[Forecast and compare](https://learning.nextia-ai.com/courses/anomaly-detection/m05/forecast-and-compare/)** · Anomaly Detection: Find What Is Unusual · Module 5, lesson 3 of 5 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** forecast each day from the days before it, alert when the real value falls outside the prediction interval, and compare five detectors on the same two years of daily orders.

| | |
|---|---|
| **Time** | About 35 minutes |
| **Needs** | An internet connection for the setup cells (less than 1 MB). No account. |
| **You should know** | Seasonal baselines, from [Normal changes with the hour and the season](https://learning.nextia-ai.com/courses/anomaly-detection/m05/normal-changes-with-the-hour-and-the-season/), and STL, from [STL residuals](https://learning.nextia-ai.com/courses/anomaly-detection/m05/stl-residuals/). The forecasting course explains forecasting models and backtesting in depth; this lesson uses one model as a tool. |
| **You do not need** | A local project. The setup below downloads everything. |
| **Tested** | Python 3.12.13 in Jupyter on macOS (Apple silicon), from a fresh start (*Restart and run all*), with the package versions of the setup cell |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/anomaly-detection/m05/forecast-and-compare/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C34/notebooks/M05-L03-forecast-and-compare/forecast-and-compare-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: pandas 3.1.1, numpy 2.5.4, scipy 1.18.1, scikit-learn 1.9.1, matplotlib 3.11.2, statsmodels 0.15.0. It needs an internet connection. The first time, it takes up to a minute.

You should see one line per package, with the same versions. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. A restart is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.1.1 numpy==2.5.4 scipy==1.18.1 scikit-learn==1.9.1 matplotlib==3.11.2 statsmodels==0.15.0
from importlib.metadata import version
for p in ['pandas', 'numpy', 'scipy', 'scikit-learn', 'matplotlib', 'statsmodels']:
    print(p, version(p))

## Setup: get the course data

Run the next cell. It downloads the data that this lesson uses (`orders.csv`, `tickets_daily.csv`, `calendar.csv`) into a folder `anomaly-course/data`, and checks each file's checksum. It needs an internet connection and no account. It takes a few seconds.

You should see `Ready:` and the files with their sizes. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `anomaly-course` (in Colab: the **Files** panel on the left), then run the cell again.

The data is made up for this course: no real people, cards or payments. [The dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C34/data/dataset-card.md) describes every table.

In [ ]:
# Setup: download the course data into anomaly-course/data/ and check each file.
# Standard library only.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C34/"
FILES = {
    "orders.csv": "bd8059fd73fe294daa157f8a6b6ae3f58319fa644eaa7ff0d31122b02cf05e30",
    "tickets_daily.csv": "36f7067db060fdd2bd084c408c0622370a012773849f243020336f1fc678b31d",
    "calendar.csv": "a5d1d5a3b640a983de72a30fca0e627a4ef23e797b1d80619e9ffcaadf118844",
}
CODE_FILES = []
HERE = Path.cwd()
# The notebook works in its own folder anomaly-course/, marked with a file, so it never writes into another project.
if Path(".nextia-notebook").exists():          # the cell ran before: start again from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
WORK = Path("anomaly-course").resolve()
(WORK / "data").mkdir(parents=True, exist_ok=True)
(WORK / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, url, path, expected=None):
    """Use a copy next to the notebook if there is one, else download it."""
    if path.exists() and (expected is None or sha256(path) == expected):
        return
    if (HERE / name).exists():
        shutil.copy(HERE / name, path)
    else:
        download(url, path)
    if expected and sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder anomaly-course and run again.")

for name, expected in FILES.items():
    fetch(name, LABS + "data/" + name, WORK / "data" / name, expected)
for name in CODE_FILES:
    fetch(name, LABS + name, WORK / name)
os.chdir(WORK)
print("Ready:", ", ".join(f"{n} ({(Path('data') / n).stat().st_size / 1e6:.1f} MB)" for n in FILES))

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

It also loads pandas as `pd`, NumPy as `np` and matplotlib as `plt`. Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it, and the chart style.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.precision", 3)
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)
warnings.filterwarnings("ignore", category=FutureWarning)
plt.rcParams.update({"figure.figsize": (9, 3.2), "axes.spines.top": False, "axes.spines.right": False})

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple, set)):
            return [norm(x) for x in (sorted(v, key=str) if isinstance(v, set) else v)]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. A forecast and an interval, by hand

A weather forecast says "between 18 and 24 degrees tomorrow". If it is 35 degrees, something unusual happened. A **prediction interval** is that range: where the real value should fall, most of the time.

Run the next cell. Five Saturdays give a forecast (their average) and a spread (how far they were from it). The sixth Saturday has 1,150 orders.

> **Check.** You should see a 99% interval of 769 to 851, and the real value outside it.

In [ ]:
# Forecast each Saturday from the Saturdays before it, and an interval from the past errors.
saturdays = np.array([800, 820, 790, 810, 830, 1150])        # the last one is the day we check
forecast = saturdays[:-1].mean()                              # the plainest forecast: the average
errors = saturdays[:-1] - forecast                            # how far the past days were from it
spread = errors.std(ddof=1)
low, high = forecast - 2.58 * spread, forecast + 2.58 * spread   # 2.58: a 99% interval for a normal error
print(f"forecast {forecast:.0f}, spread {spread:.1f}, 99% interval {low:.0f} to {high:.0f}")
print("the real value", saturdays[-1], "is", "outside" if not low <= saturdays[-1] <= high else "inside")

## 2. Larkfield's daily orders, forecast day by day

The next cell loads the daily orders and the holidays, as in the last two lessons.

In [ ]:
orders = pd.read_csv("data/orders.csv", parse_dates=["date"]).groupby("date").orders.sum()
calendar = pd.read_csv("data/calendar.csv", parse_dates=["date"]).set_index("date")
days = pd.date_range(orders.index.min(), orders.index.max(), freq="D")
orders = orders.reindex(days).astype(float)
holiday = calendar.is_holiday.reindex(days).fillna(0).astype(bool)
near_holiday = holiday | holiday.shift(1, fill_value=False) | holiday.shift(-1, fill_value=False)
after_warm_up = orders.index >= "2024-05-06"
filled = orders.interpolate(limit_direction="both")
print(len(orders), "days")

Now the forecast. We use one simple model, **SARIMA** (a seasonal ARIMA): it predicts each day from the days just before it and from the same weekday a week before. We learn its parameters on the first year only. Then the model steps through the two years, and each day's forecast uses only the days **before** it, as a live alert must.

> **Check.** You should see 16 alarm days, 15 of them on or next to a holiday. On 2025-07-05 (the heat wave), the real value is just above the interval.

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

y = np.log(filled.clip(lower=1))                                   # the log: an error of 10% is the same size on every day
first_year = y.loc["2024-04-01":"2025-03-31"]
fit = SARIMAX(first_year, order=(1, 0, 1), seasonal_order=(0, 1, 1, 7)).fit(disp=False)   # learn on the first year
model = SARIMAX(y, order=(1, 0, 1), seasonal_order=(0, 1, 1, 7)).filter(fit.params)      # then step day by day
pred = model.get_prediction(start=14, dynamic=False)                # each day from the days before it only
interval = np.exp(pred.conf_int(alpha=0.01))                        # the 99% prediction interval
low, high = interval.iloc[:, 0], interval.iloc[:, 1]
forecast = np.exp(pred.predicted_mean)
fc_alarm = pd.Series(False, index=days)
fc_alarm.loc[low.index] = (orders.loc[low.index] < low) | (orders.loc[low.index] > high)
fc_alarm &= orders.notna()
a = fc_alarm & after_warm_up
print("alarm days:", int(a.sum()), "| on or next to a holiday:", int((a & near_holiday).sum()))
print(pd.DataFrame({"orders": orders, "forecast": forecast.round(1), "low": low.round(1), "high": high.round(1)})
      .loc[["2025-07-05", "2025-11-28"]])

The chart shows June and July 2025: the interval in blue, the real orders in black, and the days outside the interval in red.

In [ ]:
part = slice("2025-06-01", "2025-07-31")
fig, ax = plt.subplots(figsize=(10, 3.4))
ax.fill_between(low.loc[part].index, low.loc[part], high.loc[part], color="lightblue", label="99% interval")
ax.plot(orders.loc[part].index, orders.loc[part], color="black", lw=1, label="orders")
out = a.loc[part]
ax.scatter(orders.loc[part].index[out], orders.loc[part][out], color="red", zorder=3, label="outside the interval")
ax.legend(loc="upper left")
plt.show()

### A common mistake: a 95% interval because it sounds strict

A 95% interval sounds safer. But by its own definition, 1 normal day in 20 falls outside it. Run the next cell to see what it does on the orders.

In [ ]:
interval95 = np.exp(pred.conf_int(alpha=0.05))                   # a 95% interval instead of 99%
a95 = pd.Series(False, index=days)
a95.loc[interval95.index] = (orders.loc[interval95.index] < interval95.iloc[:, 0]) | (orders.loc[interval95.index] > interval95.iloc[:, 1])
a95 &= orders.notna() & after_warm_up
print("95% interval: alarm days", int(a95.sum()), "| reported after marking holidays", int((a95 & ~near_holiday).sum()))

With a 95% interval, the forecast alarms on 24 days, and 7 of them are not near a holiday (1 with the 99% interval). That is fewer than 1 in 20. One likely reason: the model learned the size of its errors on a first year with holidays in it, so the interval is wide on normal days. The interval's level is a choice, like a threshold: choose it from the number of alerts the team can read.

## 3. Five detectors on the same days

> **Predict.** Which of the five detectors of this module reports the fewest days after the holidays are marked: the global threshold, the rolling median, the seasonal baseline, STL or the forecast?

Run the next cell. It builds the five detectors with the settings of the lessons, and counts their alarm days after the warm-up.

In [ ]:
def robust_z(x, center, spread):
    return (x - center) / (1.4826 * np.maximum(spread, 1e-9))

from statsmodels.tsa.seasonal import STL
fy = orders.loc["2024-04-01":"2025-03-31"]
alarms = {"global threshold": (orders - fy.mean()).abs() > 3 * fy.std()}
med = orders.shift(1).rolling(28, min_periods=14).median()
mad = (orders.shift(1) - med).abs().rolling(28, min_periods=14).median()
alarms["rolling median, 28 days"] = robust_z(orders, med, mad).abs() > 3.5
lags = pd.concat([orders.shift(7 * k) for k in (1, 2, 3, 4)], axis=1)
smed = lags.median(axis=1)
smad = lags.sub(smed, axis=0).abs().median(axis=1)
alarms["seasonal baseline, same weekday"] = robust_z(orders, smed, np.maximum(smad, 0.05 * smed)).abs() > 3.5
r = STL(filled, period=7, robust=True).fit().resid
alarms["STL remainder"] = (robust_z(r, r.median(), (r - r.median()).abs().median()).abs() > 3.5) & orders.notna()
alarms["forecast, 99% interval"] = fc_alarm
summary = pd.DataFrame({name: {"alarm days": int((f & after_warm_up).sum()),
                               "on or next to a holiday": int((f & after_warm_up & near_holiday).sum()),
                               "reported after marking holidays": int((f & after_warm_up & ~near_holiday).sum())}
                        for name, f in alarms.items()}).T
summary

The forecast reports 1 day after the holidays are marked; STL reports 75. The forecast's interval is wide (it knows that its own errors are large), so it alarms only on very big surprises, and almost all of them are holidays. Its one other day is the peak of the planted heat wave. The global threshold also reports few days, but for a bad reason: it would also miss a Tuesday that halves.

## 4. Check yourself

**Your turn 1.** Which days does the forecast report after the warm-up when the holidays and the days next to them are marked? Write them as a list of strings `"YYYY-MM-DD"` in `forecast_other_days`.

In [ ]:
forecast_other_days = ...   # your code

In [ ]:
expect_hash('forecast_other_days', forecast_other_days, 'd1c130768e2ff324')

**Your turn 2.** Build the same forecast on the daily tickets (`data/tickets_daily.csv`, all channels together): the same model, learned on the first year. List the days of the heat-wave week (2025-06-30 to 2025-07-06) that fall outside the 99% interval, as strings, in `ticket_heat_wave_alarms`.

In [ ]:
ticket_heat_wave_alarms = ...   # your code

In [ ]:
expect_hash('ticket_heat_wave_alarms', ticket_heat_wave_alarms, 'c3d3600cdf1532d1')

In the solution, the forecast flags 3 of the 7 days of the heat-wave week on the tickets. Each forecast uses the days just before it: once some high days are in the past, the next high day is less of a surprise.

## Recap

- A forecast with a prediction interval asks: "is today far from what the past says today should be?" It uses only the past, so it can run live.
- An interval that is too wide misses things; one that is too narrow alarms on normal days. The width comes from the model's own past errors.
- On Larkfield's orders, the forecast reports the fewest days after the holidays are marked. A seasonal baseline is simpler and enough when the pattern is stable.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Change points](https://learning.nextia-ai.com/courses/anomaly-detection/m05/change-points/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/anomaly-detection/m05/forecast-and-compare/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.